In [ ]:
%load_ext autoreload
%autoreload 2


In [ ]:
%matplotlib qt

In [ ]:
import requests
import urllib.request
import time
from bs4 import BeautifulSoup
#from lxml import etree

In [ ]:
import pandas as pd
idx=pd.IndexSlice

import numpy as np
import matplotlib.pyplot as plt
from pandas.plotting import register_matplotlib_converters
register_matplotlib_converters()
import matplotlib
matplotlib.rcParams.update({'font.size': 20})
from datetime import datetime, timedelta
import matplotlib.patches as patches
import seaborn as sns

In [ ]:
#import geopandas as gpd
import matplotlib.dates as mdates
import locale
import os
#from unidecode import unidecode

# Define functions

In [ ]:
def fDailyRes(iDfRes, ilReg):
 
    DfAppend=[]
    for reg in ilReg:
        for i in range(iDfRes.loc[reg].shape[0]):
            if iDfRes.loc[reg].size==4:
                print ('check fDailyRes')
                deltat=(pd.to_datetime(iDfRes.loc[reg].EndDate).to_numpy()-\
                    pd.to_datetime(iDfRes.loc[reg].StartDate).to_numpy()).astype('timedelta64[D]')
                #print (len(lDate_In))
                lDate_Out=[iDfRes.loc[reg].StartDate+timedelta(days=l+1) for l in range(int(deltat / np.timedelta64(1, 'D'))-1)]
                lCol_Out=[iDfRes.loc[reg,'Color'] for l in range(int(deltat / np.timedelta64(1, 'D'))-1)]
                lColInt_Out=[iDfRes.loc[reg,'Level'] for l in range(int(deltat / np.timedelta64(1, 'D'))-1)]
                lName_Out=[reg for l in range(int(deltat / np.timedelta64(1, 'D'))-1)]
                lEndDate_Out=[pd.to_datetime(iDfRes.loc[reg].EndDate).to_numpy() for l in range(int(deltat / np.timedelta64(1, 'D')))]
            else:
                lDate_Out=pd.date_range(start = iDfRes.loc[reg].iloc[i].StartDate, end=iDfRes.loc[reg].iloc[i].EndDate, freq='D')
                lCol_Out=[iDfRes.loc[reg,'Color'].iloc[i] for l in range(len(lDate_Out))]
                lColInt_Out=[iDfRes.loc[reg,'Level'].iloc[i] for l in range(len(lDate_Out))]
                lName_Out=[reg for l in range(len(lDate_Out))]
                lEndDate_Out=[pd.to_datetime(iDfRes.loc[reg].EndDate.iloc[i]).to_numpy() for l in range(len(lDate_Out))]
            DfAppend.append(pd.DataFrame(list(zip(lDate_Out, lEndDate_Out, lColInt_Out,lCol_Out)), 
                              columns=['StartDate','EndDate','Level','Color'], index=lName_Out))
    iDfRes=pd.concat([*DfAppend])
    iDfRes.index.name='Region'
    
    return(iDfRes)

In [ ]:
def fNamesENG(iDf):
    # adjust names for England

    for dist in iDf.drop_duplicates('sub_region_2')['sub_region_2'].values:
    
        if dist[-8:]=='District':
            if dist[:5]=='Saint':
                #print (dist,dist[5:12] )
                iDf.loc[iDf['sub_region_2']==dist,'sub_region_2']='St.'+dist[5:12]
            else:
                if dist[:-9]=='Folkestone & Hythe':
                    iDf.loc[iDf['sub_region_2']==dist,'sub_region_2']='Folkestone and Hythe'
                if (dist[:8]=='City of ') and (dist!='City of London'):
                    #print (dist, dist[8:-9])
                    iDf.loc[iDf['sub_region_2']==dist,'sub_region_2']=dist[8:-9]
                else:
                    iDf.loc[iDf['sub_region_2']==dist,'sub_region_2']=dist[:-9]
       
        if dist[:18]=='London Borough of ':#) & (dist!='nan'):
            #print (dist, dist[19:])
            iDf.loc[iDf['sub_region_2']==dist,'sub_region_2']=dist[18:]
        if dist[:11]=='Borough of ':
            #print (dist, dist[11:])
            iDf.loc[iDf['sub_region_2']==dist,'sub_region_2']=dist[11:]
        if dist[:24]=='Metropolitan Borough of ':
            if dist=='Metropolitan Borough of St Helens':
                iDf.loc[iDf['sub_region_2']==dist,'sub_region_2']='St. Helens'
            else:
                #print (dist, dist[24:])
                iDf.loc[iDf['sub_region_2']==dist,'sub_region_2']=dist[24:]
        if (dist[:8]=='City of ') and (dist!='City of London'):
            #print (dist, dist[24:])
            iDf.loc[iDf['sub_region_2']==dist,'sub_region_2']=dist[8:]
        if dist[-7:]=='Borough':
            #print (dist, dist[:-8])
            iDf.loc[iDf['sub_region_2']==dist,'sub_region_2']=dist[:-8]
        if dist[:16]=='Royal Borough of':
            iDf.loc[iDf['sub_region_2']==dist,'sub_region_2']=dist[17:]

    iDf.loc[iDf['sub_region_1']=='Borough of Halton','sub_region_1']= 'Halton'   
    iDf.loc[iDf['sub_region_1']=='Bristol City','sub_region_1']= 'Bristol'       
    iDf.loc[iDf['sub_region_2']=='Bournemouth, Christchurch and Poole','sub_region_2']= 'Bournemouth Christchurch and Poole'   
    iDf.loc[iDf['sub_region_1']=='County Durham','sub_region_1']= 'Durham'       

    for reg in lRegENG:
        if (reg not in iDf['sub_region_2'].values) & (reg in iDf['sub_region_1'].values) & (reg!='Rutland'):
            #print (reg)
            iDf.loc[(iDf['sub_region_1']==reg) & 
                 (iDf['sub_region_2']=='empty'),'sub_region_2']=reg

    return (iDf)
### IMPORTANT ###

# Restrictions -- Mobility

# Somerset West and Taunton (together) -- Taunton Deane and West Somerset (separated)
# Buckinghamshire (one) -- Decoupled into the previous four councils (2020-04-01)
# Suffolk Coastal district was merged with Waveney district on 1 April 2019 to form the new East Suffolk district.
# Somerset West and Taunton District Council is the local authority for the Somerset West and Taunton district from 1 April 2019.
# Isles of Scilly not in Google data
# Peak district national park includes different districts, I don't include it
# Rutland has missing dates in Google data

In [ ]:
def fNamesSCO(iDf):
    # adjust names for Scotland

    for dist in iDf.drop_duplicates('sub_region_1')['sub_region_1'].values:
        
        if dist[-7:]=='Council':
            iDf.loc[iDf['sub_region_1']==dist,'sub_region_1']= dist[:-8]
        if dist=='Na h-Eileanan an Iar':
            iDf.loc[iDf['sub_region_1']=='Na h-Eileanan an Iar','sub_region_1']='Comhairle nan Eilean Siar'

    return (iDf)

# missing Google mobility data for Shetland Island, Orkney and 'Comhairle nan Eilean Siar'
# for other councils also missing data for some categories

# Load data

In [ ]:
dColStr={1:'skyblue', 2:'gold', 3:'darkorange',4:'red',5:'purple', 6:'grey'}

## NPIs Data

### ITA

In [ ]:
dNameReg={'Friuli Venezia Giulia':'Friuli-Venezia Giulia', 'Lombardia':'Lombardy', 'Piemonte':'Piedmont',
 'Puglia':'Apulia', 'Sardegna':'Sardinia', 'Sicilia':'Sicily', 'Toscana':'Tuscany', "Valle d'Aosta":'Aosta'}

DfResITA=pd.read_csv('../../Data/Mobility_Response/Restrictions/ITA/Catta.csv')
DfResITA=DfResITA.rename(columns={'data':'Date','denominazione_regione':'Region','colore':'Color'})

dColor={'bianco':'skyblue','giallo':'gold','arancione':'darkorange','rosso':'red'}
dLevel={'skyblue':1,'gold':2,'darkorange':3,'red':4}

DfResITA['Color']= [dColor[colita] for colita in DfResITA['Color'].values]
DfResITA['Level']= [dLevel[colita] for colita in DfResITA['Color'].values]
DfResITA=DfResITA[DfResITA['Date']<'2022-04-01']
DfResITA['Date']=[datetime.fromisoformat(val) for val in DfResITA['Date'].values]
DfResITA.loc[DfResITA['Region']=='Provincia autonoma Bolzano','Region']='A.P. Bolzano'
DfResITA.loc[DfResITA['Region']=='Provincia autonoma Trento','Region']='A.P. Trento'

for reg in list(dNameReg.keys()):
    DfResITA.loc[DfResITA['Region']==reg,'Region']=dNameReg[reg]

DfResITA=DfResITA.sort_values(by=['Region','Date'])
lRegITA=list(DfResITA.drop_duplicates('Region')['Region'].values)


In [ ]:
for reg in lRegITA:
    if DfResITA.set_index('Region').loc[reg].shape[0]!=pd.date_range(start = DfResITA.iloc[0].Date, end=DfResITA.iloc[-1].Date, freq='D').shape[0]:
        print ('Warning: missind days')
if len(DfResITA.drop_duplicates(['Region','Date']))!= len(DfResITA):
    print ('Warning: duplicated entries')

### ENG

In [ ]:
dColStr

In [ ]:
DfResENG= pd.read_csv('../../Data/Mobility_Response/Restrictions/ENG/England.csv', usecols=[1,2,3,4],parse_dates=['EndDate'])
DfResENG=DfResENG.set_index(['Region','StartDate']).sort_index()
DfResENG['EndDate']=DfResENG.EndDate.values - np.timedelta64(1,'D')

DfResENG=DfResENG.loc[idx[:,'2020-10-14':],].reset_index()
DfResENG['Color']=[dColStr[tier] for tier in DfResENG['Level'].values]
lRegENG=list(DfResENG.drop_duplicates('Region')['Region'].values)
DfResENG=DfResENG.set_index('Region')
DfResENG.sort_index(inplace=True)

DfResENG=fDailyRes(DfResENG,lRegENG)
DfResENG=DfResENG.reset_index().rename(columns={'StartDate':'Date'}).drop(columns='EndDate')
DfResENG=DfResENG[['Region', 'Date', 'Color', 'Level']]


In [ ]:
for reg in lRegENG:
    if DfResENG.set_index('Region').loc[reg].shape[0]!=pd.date_range(start = DfResENG.iloc[0].Date, end=DfResENG.iloc[-1].Date, freq='D').shape[0]:
        print ('Warning: missind days')
if len(DfResENG.drop_duplicates(['Region','Date']))!= len(DfResENG):
    print ('Warning: duplicated entries')

### SCO

In [ ]:
dColStr

In [ ]:
DfResSCO= pd.read_csv('../../Data/Mobility_Response/Restrictions/SCO/Scotland.csv').dropna()
DfResSCO.loc[:,'Level']=[int(lev[6:])+1 for lev in DfResSCO['Level'].values]
DfResSCO['Color']=[dColStr[lev] for lev in DfResSCO['Level']]
lRegSCO=list(DfResSCO.drop_duplicates('Region')['Region'].values)
DfResSCO.set_index('Region', inplace=True)

DfResSCO= fDailyRes(DfResSCO,lRegSCO)
DfResSCO=DfResSCO.reset_index().rename(columns={'StartDate':'Date'}).drop(columns='EndDate')
DfResSCO=DfResSCO[['Region', 'Date', 'Color', 'Level']]

In [ ]:
for reg in lRegSCO:
    if DfResSCO.set_index('Region').loc[reg].shape[0]!=pd.date_range(start = DfResSCO.iloc[0].Date, end=DfResSCO.iloc[-1].Date, freq='D').shape[0]:
        print ('Warning: missinG days')
if len(DfResSCO.drop_duplicates(['Region','Date']))!= len(DfResSCO):
    print ('Warning: duplicated entries')

### CA

In [ ]:
dColStr

In [ ]:
dNewLevel={4:1,3:2, 2:3, 1:4}
DfResCA=pd.read_csv('../../Data/Mobility_Response/Restrictions/CA/California.csv', index_col=0)

for c in range(len(DfResCA.columns)):
    DfResCA[DfResCA.columns[c]]=[dNewLevel[oldLev] for oldLev in DfResCA[DfResCA.columns[c]].values]
    if c<len(DfResCA.columns)-1:
        for date in pd.date_range(start = DfResCA.columns[c], end=DfResCA.columns[c+1], freq='D')[1:-1]:
            DfResCA[date.strftime("%Y.%m.%d")]=DfResCA[DfResCA.columns[c]]

DfResCA=DfResCA.stack().to_frame().rename_axis(['Region','Date']).rename(columns={0:'Level'})
DfResCA['Color']=[dColStr[tier] for tier in DfResCA['Level'].values]
DfResCA=DfResCA.sort_index().reset_index()
DfResCA['Date']=pd.to_datetime(DfResCA['Date'])
DfResCA.set_index(['Region','Date'], inplace=True)
#load data for days when stay-at-home mandates were put in place
DfResCA_NL=pd.read_csv('../../Data/Mobility_Response/Restrictions/CA/CaliforniaNL.csv',dtype={'Region':'str','StartDate':'str',
                            'EndDate':'str','Level':'int'}, parse_dates=['StartDate','EndDate'], usecols=[1,2,3,4])
# adjust for Stay-at-home order
for reg in DfResCA_NL.Region:
    datein=DfResCA_NL.set_index('Region').loc[reg].StartDate.strftime("%Y-%m-%d")
    dateout=DfResCA_NL.set_index('Region').loc[reg].EndDate.strftime("%Y-%m-%d")

    DfResCA.loc[idx[reg,datein:dateout],'Level']=5
    DfResCA.loc[idx[reg,datein:dateout],'Color']=dColStr[5]
    
DfResCA.reset_index(inplace=True)
DfResCA=DfResCA[DfResCA['Region']!='Alpine']
DfResCA=DfResCA[DfResCA['Region']!='Sierra']
lRegCA=DfResCA.drop_duplicates('Region')['Region'].values


In [ ]:
for reg in lRegCA:
    if DfResCA.set_index('Region').loc[reg].shape[0]!=pd.date_range(start = DfResCA.iloc[0].Date, end=DfResCA.iloc[-1].Date, freq='D').shape[0]:
        print ('Warning: missind days')
if len(DfResCA.drop_duplicates(['Region','Date']))!= len(DfResCA):
    print ('Warning: duplicated entries')

In [ ]:
DfResCA.to_csv('../../Data//NPI/CA/CA_NPI.csv')

### CHL

In [ ]:
DfTable=pd.read_csv('../../Data/Mobility_Response/Restrictions/CHL/Comunas_PasoZone.csv', usecols=[0])


In [ ]:

# load file from Government
dNewLev={1:5, 2:4, 3:3, 4:2, 5:1}
DfResCHL=pd.read_csv('../../Data/Mobility_Response/Restrictions/CHL/Chile.csv',usecols=[2,3,4,5,6])
DfResCHL=DfResCHL.rename(columns={'comuna_residencia':'Comuna','Fecha':'Date','Paso':'Level', 'codigo_comuna':'Code'})
DfResCHL=DfResCHL[DfResCHL['Date']<='2021-09-30']
DfResCHL['Level']=DfResCHL['Level'].astype(int)

###
# print to file the comunas with sub-comuna level of NPI
lDiffPaso=[]
for com in DfResCHL.drop_duplicates(['Comuna','zona'])[DfResCHL.drop_duplicates(['Comuna','zona']
            )['zona']!='Total'].drop_duplicates('Comuna')['Comuna']:#.set_index(['Comuna','zona'])#.unstack()

    Dftemp=DfResCHL.set_index(['Comuna','zona','Date']).loc[com].unstack('zona')['Level']
    Dftemp=Dftemp[Dftemp[Dftemp.columns[0]]!=Dftemp[Dftemp.columns[1]]]
    
    lDiffPaso.append([com,Dftemp.shape[0],list(Dftemp.columns)])
    
Df=pd.DataFrame(lDiffPaso, columns=['Comuna','Different paso', 'Zone']).sort_values('Comuna')
Df.set_index('Comuna').to_csv('../../Data/Mobility_Response/Restrictions/CHL/Comunas_PasoZone.csv')

# number of comunas with sub-comuna application of the NPIs is 36/346
#####

DfResCHL['Date']=pd.to_datetime(DfResCHL.Date.values).to_numpy()
DfResCHL['Level']=DfResCHL['Level'].astype(int)
#DfResCHL_D1.loc[DfResCHL_D1['zona']=='Rural','Comuna']=DfResCHL_D1.loc[DfResCHL_D1['zona']=='Rural','Comuna']+'_R'
#DfResCHL_D1.loc[DfResCHL_D1['zona']=='Urbana','Comuna']=DfResCHL_D1.loc[DfResCHL_D1['zona']=='Urbana','Comuna']+'_U'
#DfResCHL['Comuna']=[unidecode(com) for com in DfResCHL['Comuna'].values]

# first transform old level into new level
DfResCHL['Level']=[dNewLev[oldLev] for oldLev in DfResCHL['Level'].values]
DfResCHL['Color']=[dColStr[colint] for colint in DfResCHL['Level'].values]
DfResCHL=DfResCHL[DfResCHL['zona']=='Total'].drop(columns='zona')

"""
DfResCHL_D1.set_index('Comuna', inplace=True)
DfResCHL_D1=DfResCHL_D1.loc[['Isla de Pascua','Juan Fernandez','Santo Domingo','Tucapel', 
                             'Villarrica']].drop(
    columns=['zona']).reset_index()
"""

DfResCHL=DfResCHL.rename(columns={'Comuna':'Region'})
DfResCHL.sort_values('Region', inplace=True)
lRegCHL= DfResCHL.drop_duplicates('Region')['Region'].tolist()

# load code comuna-province-region
DfCHLCode=pd.read_excel('../../Data/Mobility_Response/Mobility/Chile/PopComuna.xlsx', usecols=[2,3,4,5])
DfCHLCode= DfCHLCode.drop_duplicates('Nombre Comuna').rename(columns={'Nombre Comuna':'Region', 'Comuna':'Code',
                                        'Provincia':'Code Prov','Nombre Provincia':'Provincia'})

DfResCHL=DfResCHL.merge(DfCHLCode.drop(columns='Region'), on='Code', how='inner')

### Ont

In [ ]:
dNamesOnt={'Brantford':'Brant',
           'Bruce':'Grey Bruce', 
           'Cochrane':'Porcupine',
           'Dufferin':'Wellington-Dufferin-Guelph', 
           'Elgin':'Southwestern',
           'Essex':'Windsor-Essex', 
           'Frontenac':'Kingston, Frontenac Lennox Addington',
           'Greater Sudbury':'Sudbury', 
           'Grey':'Grey Bruce', 
           'Haldimand':'Haldimand-Norfolk',
           'Haliburton':'Haliburton, Kawartha, Pine Ridge',
           'Hastings':'Hastings Prince Edward',
           'Huron':'Huron Perth', 
           'Kawartha Lakes':'Haliburton, Kawartha, Pine Ridge',
           'Kenora':'Northwestern',
           'Lanark':'Leeds, Grenville Lanark', 
           'Leeds and Grenville':'Leeds, Grenville Lanark',
           'Lennox and Addington':'Kingston, Frontenac Lennox Addington',
           'Manitoulin':'Sudbury',
           'Middlesex':'Middlesex-London',
           'Muskoka':'Simcoe Muskoka',
           'Simcoe':'Simcoe Muskoka',
           'Nipissing': 'North Bay Parry Sound',
           'Norfolk':'Haldimand-Norfolk',
           'Northumberland':'Haliburton, Kawartha, Pine Ridge',
           'Oxford':'Southwestern',
           'Parry Sound':'North Bay Parry Sound', 
           'Perth':'Huron Perth',
           'Prescott and Russell':'Eastern Ontario', 
           'Prince Edward':'Hastings Prince Edward',
           'Rainy River': 'Northwestern',
           'Stormont, Dundas and Glengarry':'Eastern Ontario',
           'Wellington':'Wellington-Dufferin-Guelph',
           'Waterloo':'Waterloo'}


In [ ]:
dColStr

In [ ]:
DfResOnt=pd.read_csv('../../Data/Mobility_Response/Restrictions/Ont/response_framework.csv', usecols=[0,2,3,4])
DfResOnt.columns=['Region','Level','StartDate','EndDate']
DfResOnt['Region']=[' '.join([test for test in t.split() if test not in ['Public', 'Health', 'Unit',
                    'Services','County','District','Region','Department','of','and','&','United','Counties']]) 
                    for t in DfResOnt['Region'].values]
dColStrOnt={'Prevent':'skyblue', 'Protect':'gold', 'Restrict':'darkorange', 'Control':'red',
         'Lockdown':'purple', 'Shutdown':'grey','Other':'white'}
DfResOnt['Color']=[dColStrOnt[colint] for colint in DfResOnt['Level'].values]
dNPIint={'Prevent':1, 'Protect':2, 'Restrict':3, 'Control':4,
         'Lockdown':5, 'Shutdown':6,'Other':7}
DfResOnt['Level']=[dNPIint[colint] for colint in DfResOnt['Level'].values]
lRegOnt=DfResOnt.reset_index().drop_duplicates('Region')['Region'].values
DfResOnt=DfResOnt.set_index('Region').sort_index()[['StartDate','EndDate', 'Level', 'Color']]

DfResOnt=fDailyRes(DfResOnt,lRegOnt)
DfResOnt.reset_index(inplace=True)

# PHU in Ontario bigger than google mobility data
DfResOnt_t=pd.DataFrame()
for reg in list(dNamesOnt.keys()):
    dftemp=DfResOnt[DfResOnt['Region']==dNamesOnt[reg]].copy()
    dftemp.loc[dftemp['Region']==dNamesOnt[reg],'Region']=reg
    DfResOnt_t=pd.concat([DfResOnt_t,dftemp])
    
# drop those PHU that I decouple, keep the names of Brant PHU and Sudbury
for reg in list(dNamesOnt.values()):
    if reg not in ['Brant','Sudbury']:
        #print (reg)
        DfResOnt.drop(DfResOnt.loc[DfResOnt['Region']==reg].index, inplace=True)
    
#DfResOnt= pd.concat([DfResOnt,DfResOnt_t])
DfResOnt=DfResOnt.reset_index().rename(columns={'StartDate':'Date'}).drop(columns='EndDate')
#DfResOnt=DfResOnt[['Region', 'Date', 'Color', 'Level']].sort_values(['Region','Date'])
#DfResOnt.loc[DfResOnt['Region']=='Waterloo,', 'Region']= 'Waterloo'
#lRegOnt=DfResOnt.reset_index().drop_duplicates('Region')['Region'].values



In [ ]:
# Check
for reg in lRegOnt:
    if DfResOnt.set_index('Region').loc[reg].shape[0]!=pd.date_range(start = DfResOnt.iloc[0].Date, end=DfResOnt.iloc[-1].Date, freq='D').shape[0]:
        print ('Warning: missind days')
if len(DfResOnt.drop_duplicates(['Region','Date']))!= len(DfResOnt):
    print ('Warning: duplicated entries')

### ZAF

In [ ]:
dColStr

In [ ]:
DfResZAF= pd.read_csv('../../Data/Mobility_Response/Restrictions/ZAF/SouthAfrica.csv', index_col=0)
DfResZAF['Color']=[dColStr[colint] for colint in DfResZAF['Level'].values]
lRegZAF=DfResZAF.reset_index().drop_duplicates('Region')['Region'].values
DfResZAF=fDailyRes(DfResZAF,lRegZAF)
DfResZAF=DfResZAF.rename(columns={'StartDate':'Date'}).drop(columns='EndDate')
DfResZAF=DfResZAF[['Date', 'Color', 'Level']]

DfResZAF.reset_index(inplace=True)


In [ ]:
# Check
if DfResZAF.shape[0]!=pd.date_range(start = DfResZAF.iloc[0].Date, end=DfResZAF.iloc[-1].Date, freq='D').shape[0]:
    print ('Warning: missind days')
if len(DfResZAF.drop_duplicates(['Region','Date']))!= len(DfResZAF):
    print ('Warning: duplicated entries')

## Mobility

### Google Data

In [ ]:
# Google mobility Data
lDfGoogle=[]
for c, cnt in enumerate(['IT','CL','US','GB','GB','CA','ZA']):
    #print (cnt)
    ltemp=[]
    for year in ['2020','2021','2022']:
        ltemp.append(pd.read_csv('../../Data/Mobility_Response/Mobility/Google/'+year+'_'+cnt+'_Region_Mobility_Report.csv',
                     usecols=[2,3,8,9,10,11,12,13,14],quotechar = '"', parse_dates=['date']))
     
    lDfGoogle.append(pd.concat([temp for temp in ltemp]))
    
    # Italy
    if c==0:
        lDfGoogle[c]=lDfGoogle[c].fillna('empty')
        lDfGoogle[c].loc[lDfGoogle[c]['sub_region_2']=='Autonomous Province of Bolzano – South Tyrol',
                 'sub_region_1']='A.P. Bolzano'
        lDfGoogle[c].loc[lDfGoogle[c]['sub_region_2']=='Autonomous Province of Bolzano – South Tyrol',
                 'sub_region_2']='A.P. Bolzano'
        lDfGoogle[c].loc[lDfGoogle[c]['sub_region_2']=='Autonomous Province of Trento',
                 'sub_region_1']='A.P. Trento'
        lDfGoogle[c].loc[lDfGoogle[c]['sub_region_2']=='Autonomous Province of Trento',
                 'sub_region_2']='A.P. Trento'
        
        lDfGoogle[c].loc[lDfGoogle[c]['sub_region_2']=='empty',
                 'sub_region_2']=lDfGoogle[c].loc[lDfGoogle[c]['sub_region_2']=='empty',
                 'sub_region_1'].values
        
        lDfGoogle[c]=lDfGoogle[c].drop(columns='sub_region_1').rename(columns={'sub_region_2':'Region'})
    # Chile
    if c==1:
        # this is for Chile but I don't use mobility data from google in this case. 
        # I keep it in case we use want to use for SA
        lDfGoogle[c]=lDfGoogle[c].drop(columns='sub_region_1').dropna().rename(columns={'sub_region_2':'Region'})
        lDfGoogle[c]['Region']=[' '.join([test for test in t.split() if test not in ['Province']]) for t in lDfGoogle[1]['Region'].values]

    # California
    if c==2:
        lDfGoogle[c]=lDfGoogle[2][lDfGoogle[2]['sub_region_1']=='California'].dropna(subset=['sub_region_2'])
        lDfGoogle[c]['sub_region_2']=[reg[:-7] for reg in lDfGoogle[c]['sub_region_2'].values]
        lDfGoogle[c]=lDfGoogle[c].drop(columns='sub_region_1').rename(columns={'sub_region_2':'Region'})
    
    # England
    if c==3:
        lDfGoogle[c]=lDfGoogle[c].fillna('empty')
        # adjust names for England
        lDfGoogle[c]=fNamesENG(lDfGoogle[c])
        lDfGoogle[c]= lDfGoogle[c].drop(columns='sub_region_1').rename(columns={'sub_region_2':'Region'})
    
    # Scotland
    if c==4:
        lDfGoogle[c]=lDfGoogle[c].dropna(subset=['sub_region_1'])
        lDfGoogle[c]=fNamesSCO(lDfGoogle[c])
        lDfGoogle[c]= lDfGoogle[c].drop(columns='sub_region_2').rename(columns={'sub_region_1':'Region'})
    
    # Ontario
    if c==5:
        lDfGoogle[c]=lDfGoogle[c].set_index('sub_region_1').loc['Ontario'].dropna(
            subset=['sub_region_2']).reset_index(drop=True)
        lDfGoogle[c]['sub_region_2']=[' '.join([test for test in t.split() if test not in ['County', 
                    'District', 'Department','Services','United', 'Counties','Regional','Municipality','of']]) 
                    for t in lDfGoogle[c]['sub_region_2'].values]
    # South Africa
    if c==6:
        lDfGoogle[c]=lDfGoogle[c].dropna(subset=['sub_region_1']).drop(columns=['sub_region_2']
                                    ).rename(columns={'sub_region_1':'Region','date':'Date'})
    print (c)
    lDfGoogle[c].columns=['Region','Date', 'Retail', 'Grocery', 'Parks', 'Stations',
                          'Workplaces', 'Residential']
    #lDfGoogle[c]['Date']=[datetime.fromisoformat(val) for val in lDfGoogle[c]['Date'].values]



### Telefonica Data

#### CHL

In [ ]:
DfMobCHL=pd.read_csv('../../Data/Mobility_Response/Mobility/Chile/trips_time_mean_by_date_comuna.csv',
                    parse_dates=['date'])
DfMobCHL=DfMobCHL.rename(columns={'geo_area_id':'Code', 'Comuna':'Region','date':'Date'})
DfMobCHL['Date']=pd.to_datetime(DfMobCHL.Date.values).to_numpy()

# check for days with extremely low number of devices
DfCheck=DfMobCHL.set_index(['Code','Date','home_is_work']).groupby(level='Date').sum().reset_index()
lDropDate=DfCheck[DfCheck['total_devices']<DfCheck['total_devices'].mean()-2.5*\
                  DfCheck.std()['total_devices']].Date
# drop those days
DfMobCHL= DfMobCHL.set_index('Date').drop(labels=lDropDate).reset_index()

"""
### in the case we aggregate over the two groups home_is_work
# aggregate for total devices in each comuna
DfDevices= DfMobCHL.groupby(['Date','Code']).sum()[['total_devices']].reset_index()
# aggregate over the dimension home is work
DfMobCHL= DfMobCHL.groupby(['Date','Code']).mean()[['time_spent_residential','trips_workplace']].reset_index()
# merge for keeping the number of devices data
DfMobCHL= DfMobCHL.merge(DfDevices, on=['Date','Code'], how='inner')
###
"""

# keep only when home!= work
DfMobCHL=DfMobCHL[DfMobCHL['home_is_work']==False]
# The day of the week with Monday=0, Sunday=6.
DfMobCHL['WeekDay']= DfMobCHL['Date'].dt.dayofweek
# column for the year only, used in calculating the portion of devices compared to the entire population
DfMobCHL['Year']=[date.year for date in DfMobCHL['Date']]

# data population at the comuna level (annual)
DfPop=pd.read_excel('../../Data/Mobility_Response/Mobility/Chile/PopComuna.xlsx')
DfPop=DfPop[['Comuna', 'Sexo (1=Hombre 2=Mujer)','Area (1=Urbano 2=Rural)','Grupo edad' ,
       'Poblacion 2020','Poblacion 2021']].set_index(
    ['Comuna','Sexo (1=Hombre 2=Mujer)','Area (1=Urbano 2=Rural)','Grupo edad'])
DfPop=DfPop.groupby(level='Comuna').sum()
DfPop=DfPop.rename(columns={'Poblacion 2020':2020, 'Poblacion 2021':2021})
DfPop=DfPop.stack().to_frame('Population').rename_axis(['Code','Year']).reset_index()

# merge pop into main Df
DfMobCHL=DfMobCHL.merge(DfPop, on=['Code','Year'], how='inner')

DfMobCHL['Coverage']=DfMobCHL['total_devices'].div(DfMobCHL['Population'])*100
# drop comunas-days with coverage of less than 1% or more than 100%
Dftemp=DfMobCHL[['Code', 'Coverage']].groupby(['Code']).mean()
lDropComuna=list(Dftemp[(Dftemp['Coverage']>100) | (Dftemp['Coverage']<1)].index)
DfMobCHL=DfMobCHL.set_index('Code').drop(labels=lDropComuna).reset_index()

In [ ]:
DfMobCHL=pd.read_csv('../../Data/Mobility_Response/Mobility/Chile/trips_time_mean_by_date_comuna.csv',
                    parse_dates=['date'])
DfMobCHL=DfMobCHL.rename(columns={'geo_area_id':'Code', 'Comuna':'Region','date':'Date'})
DfMobCHL['Date']=pd.to_datetime(DfMobCHL.Date.values).to_numpy()

# check for days with extremely low number of devices
DfCheck=DfMobCHL.set_index(['Code','Date','home_is_work']).groupby(level='Date').sum().reset_index()
lDropDate=DfCheck[DfCheck['total_devices']<DfCheck['total_devices'].mean()-2.5*\
                  DfCheck.std()['total_devices']].Date
# drop those days
DfMobCHL= DfMobCHL.set_index('Date').drop(labels=lDropDate).reset_index()

"""
### in the case we aggregate over the two groups home_is_work
# aggregate for total devices in each comuna
DfDevices= DfMobCHL.groupby(['Date','Code']).sum()[['total_devices']].reset_index()
# aggregate over the dimension home is work
DfMobCHL= DfMobCHL.groupby(['Date','Code']).mean()[['time_spent_residential','trips_workplace']].reset_index()
# merge for keeping the number of devices data
DfMobCHL= DfMobCHL.merge(DfDevices, on=['Date','Code'], how='inner')
###
"""

# keep only when home!= work
DfMobCHL=DfMobCHL[DfMobCHL['home_is_work']==False]
# The day of the week with Monday=0, Sunday=6.
DfMobCHL['WeekDay']= DfMobCHL['Date'].dt.dayofweek
# column for the year only, used in calculating the portion of devices compared to the entire population
DfMobCHL['Year']=[date.year for date in DfMobCHL['Date']]

# data population at the comuna level (annual)
DfPop=pd.read_excel('../../Data/Mobility_Response/Mobility/Chile/PopComuna.xlsx')
DfPop=DfPop[['Comuna', 'Sexo (1=Hombre 2=Mujer)','Area (1=Urbano 2=Rural)','Grupo edad' ,
       'Poblacion 2020','Poblacion 2021']].set_index(
    ['Comuna','Sexo (1=Hombre 2=Mujer)','Area (1=Urbano 2=Rural)','Grupo edad'])
DfPop=DfPop.groupby(level='Comuna').sum()
DfPop=DfPop.rename(columns={'Poblacion 2020':2020, 'Poblacion 2021':2021})
DfPop=DfPop.stack().to_frame('Population').rename_axis(['Code','Year']).reset_index()

# merge pop into main Df
DfMobCHL=DfMobCHL.merge(DfPop, on=['Code','Year'], how='inner')
DfMobCHL['Coverage']=DfMobCHL['total_devices'].div(DfMobCHL['Population'])*100
# drop comunas-days with coverage of less than 1% or more than 100%
Dftemp=DfMobCHL[['Code', 'Coverage']].groupby(['Code']).mean()
lDropComuna=list(Dftemp[(Dftemp['Coverage']>100) | (Dftemp['Coverage']<1)].index)
DfMobCHL=DfMobCHL.set_index('Code').drop(labels=lDropComuna).reset_index()

DfMobCHL=DfMobCHL.set_index(['Date','Code','WeekDay'])[['trips_workplace','time_spent_residential']].rename(
    columns={'trips_workplace':'Workplaces', 'time_spent_residential':'Residential'})

# Baseline separately for each day of the week
DfBaseCHL=DfMobCHL.reset_index().set_index('Date').sort_index().loc[idx['2020-03-02':'2020-03-15'],].reset_index().copy()
DfBaseCHL=DfBaseCHL.groupby(['Code','WeekDay']).mean(numeric_only=True).rename(
            columns={'Residential':'Residential_Base', 'Workplaces':'Workplaces_Base'})
DfMobCHL=DfMobCHL.reset_index().merge(DfBaseCHL.reset_index(), on=['Code', 'WeekDay'], how='inner')

DfMobCHL['Residential_Change']=(DfMobCHL['Residential'].div(DfMobCHL['Residential_Base'])-1)*100
DfMobCHL['Workplaces_Change']=(DfMobCHL['Workplaces'].div(DfMobCHL['Workplaces_Base'])-1)*100


In [ ]:
# get df for aggregating at the province level
DfBaseCHLProv=DfMobCHL.set_index(['Date','Code','WeekDay']).sort_index().loc[idx['2020-03-02':'2020-03-15',:,:],['Workplaces','Residential']].reset_index().copy()
DfMobCHLProv=DfMobCHL.merge(DfResCHL[['Code','Provincia']].drop_duplicates('Code'), on='Code', how='inner').drop(
    columns=['Workplaces_Base','Residential_Base','Residential_Change','Workplaces_Change'])

# Baseline province
DfBaseCHLProv=DfBaseCHLProv.merge(DfResCHL[['Code','Provincia']].drop_duplicates('Code'), on='Code', 
                                          how='inner').drop(columns=['Code'])
DfBaseCHLProv=DfBaseCHLProv.groupby(['Date','WeekDay','Provincia']).mean().reset_index()
DfBaseCHLProv=DfBaseCHLProv.groupby(['Provincia','WeekDay']).mean(numeric_only=True)

DfMobCHLProv=DfMobCHLProv.groupby(['Date','WeekDay','Provincia']).mean().drop(columns=['Code'])
DfMobCHLProv=((DfMobCHLProv.div(DfBaseCHLProv)-1)*100).reset_index()

DfMobCHLProv.set_index('Provincia').drop(columns='WeekDay').to_csv('../../Data/Mobility_Response/MobTelCHLProv.csv')


In [ ]:
DfMobCHL=DfMobCHL.drop(columns=['Residential', 'Residential_Base', 'Workplaces', 'Workplaces_Base']).rename(columns={'Residential_Change':'Residential',
                                                                                                           'Workplaces_Change':'Workplaces'})

In [ ]:

fig, subfig= plt.subplots(1, figsize=(12,10))
sns.histplot(data=DfMobCHL, hue='home_is_work', x='total_devices',multiple='dodge', ax=subfig,
             log_scale=True, bins=50)
#sns.histplot(data=DfMobCHL, hue='home_is_work', x='Residential', ax=subfig[1], bins=50)

fig.tight_layout()
#fig.savefig('../Figures/Adherence/CHL/HomeWork_Devices.png')


In [ ]:

# Figure coverage
fig, subplot= plt.subplots(figsize=(10,10))
sns.histplot(x='Coverage', data=DfMobCHL, ax=subplot)
subplot.set_xlabel('Devices coverage (%)')
#fig.savefig('../Figures/Adherence/CHL/Hist_PopCoverageDevices.png')


In [ ]:
Df=DfMobCHL.groupby('Date').mean(numeric_only=True).copy()

fig, subfig= plt.subplots(1, figsize=(10,6))

sns.histplot(DfMobCHL['Residential'], bins=50)
sns.histplot(DfMobCHL['ResidentialOther'], bins=50)

#subfig.axhline(DfBaseCHL.mean()['Residential'], linestyle='--', c='r')
#subfig.fill_between(Df.index.values, DfBaseCHL.min()['Residential'], DfBaseCHL.max()['Residential'], alpha=0.3)
subfig.tick_params('x', rotation=45)
subfig.set_xlim(-100,100)

### Incidence

#### ITA

In [ ]:
# Italy
dicRegITA={'abruzzo':'Abruzzo', 'basilicata':'Basilicata', 'calabria':'Calabria', 'campania':'Campania', 
           'emilia_romagna':'Emilia-Romagna','friuli_venezia_giulia':'Friuli-Venezia Giulia', 'lazio':'Lazio',
           'liguria':'Liguria', 'lombardia':'Lombardy', 'marche':'Marche', 'molise':'Molise', 
           'pa_bolzano':'A.P. Bolzano', 'pa_trento':'A.P. Trento', 'piemonte':'Piedmont', 'puglia':'Apulia',
           'sardegna':'Sardinia', 'sicilia':'Sicily', 'toscana':'Tuscany', 'umbria':'Umbria',
           'valle_daosta':'Aosta', 'veneto':'Veneto'}
lDf=[]
for sFile in os.listdir('../Data/Adherence_Paper/Incidence/ITA/Cases/'):
    Dftemp=pd.read_csv('../Data/Adherence_Paper/Incidence/ITA/Cases/'+sFile, usecols=[0,1], parse_dates=['data'])
    Dftemp['Region']=dicRegITA[sFile[11:-13]]
    lDf.append(Dftemp)
    
DfIncITA= pd.concat(lDf)
DfIncITA.columns=['Date','New cases','Region']
DfIncITA['Year'] = [date.year for date in DfIncITA['Date']]

### Population ###

dicRegITA={ 'Lombardia':'Lombardy', 'Piemonte':'Piedmont', 'Puglia':'Apulia',
           'Sardegna':'Sardinia', 'Sicilia':'Sicily', 'Toscana':'Tuscany',
          'Provincia Autonoma Bolzano / Bozen': 'A.P. Bolzano', 'Provincia Autonoma Trento':'A.P. Trento'}

lDf=[]
for year in [2020,2021,2022]:
    lDf.append(pd.read_csv('../Data/Adherence_Paper/Incidence/ITA/Pop/POP_ITA_'+str(year)+'.csv',
                           usecols=[3,12,13], sep=';'))
    
DfPopITA=pd.concat(lDf)
for sReg in dicRegITA.keys():
    DfPopITA.loc[DfPopITA['Territorio']==sReg,'Territorio']=dicRegITA[sReg]
DfPopITA.columns= ['Region','Year','Population']

### merge to incidence ###

DfIncITA = DfIncITA.merge(DfPopITA, on=['Region','Year'], how='inner')
DfIncITA['Incidence']=DfIncITA['New cases'].div(DfIncITA['Population'])

#### CA

In [ ]:
# California
DfIncCA= pd.read_csv('../Data/Adherence_Paper/Incidence/CA/Pop_Cases_Deaths_CA.csv', usecols=[1,2,4,5]).dropna()
DfIncCA.columns=['Date','Region','Population','New cases']
DfIncCA['Date']= pd.to_datetime(DfIncCA['Date'])
DfIncCA['Incidence']=DfIncCA['New cases'].div(DfIncCA['Population'])
#DfIncCA

#### SCO

In [ ]:
# Scotland
DfIncSCO=pd.read_csv('../Data/Adherence_Paper/Incidence/SCO/SCO_Cases.csv', parse_dates=['Date'], usecols=[0,2,3])
DfIncSCO.columns=['Date','Region','New cases']
DfIncSCO['Year']= [date.year for date in DfIncSCO['Date']]
DfIncSCO=DfIncSCO[DfIncSCO['Year']<=2021]

DfPopSCO=pd.read_excel('../Data/Adherence_Paper/Incidence/SCO/SCO_Pop_trend.xlsx', sheet_name='Table_1', header=5,
                      usecols=[1,2,3,4])
DfPopSCO=DfPopSCO[DfPopSCO['Sex']=='Persons'].drop(columns='Sex')
DfPopSCO.columns=['Region','Year','Population']
DfPopSCO=DfPopSCO[(DfPopSCO['Year']>=2020)]

DfIncSCO=DfIncSCO.merge(DfPopSCO, on=['Region','Year'], how = 'inner')
DfIncSCO=DfIncSCO.drop(columns='Year')
DfIncSCO['Incidence']=DfIncSCO['New cases'].div(DfIncSCO['Population'])

dicTemp= {'City of Edinburgh':'Edinburgh','Na h-Eileanan Siar':'Comhairle nan Eilean Siar', 
          'Orkney Islands':'Orkney' }
for sReg in dicTemp.keys():
    DfIncSCO.loc[DfIncSCO['Region']==sReg,'Region']= dicTemp[sReg]

#### CHL

In [ ]:
# incidence data from MINSAL github repository, incomplete,no data for every day, dataset reduced to 1/4 of original
DfIncCHL=pd.read_csv('../../Data/Mobility_Response/Incidence/CHL/datos-covid-19/producto1/Covid-19_std.csv',
                     usecols=[2,3,4,5,6],parse_dates=['Fecha'], dayfirst=True, dtype={'codigo comuna': int}
                    ).dropna()

DfIncCHL.columns = ['Region','Code','Population','Date', 'Cases']
DfIncCHL = DfIncCHL.astype({'Code': int})

lDfIncCHL= []
for sArea in DfIncCHL.drop_duplicates('Region').Region:
    Dftemp=DfIncCHL.set_index('Region').loc[sArea].sort_values('Date')
    aNewCases=Dftemp['Cases'].iloc[1:]-Dftemp['Cases'].iloc[:-1].values
    aDays= (pd.to_datetime(Dftemp['Date'].iloc[1:])-pd.to_datetime(Dftemp['Date'].iloc[:-1]))
    aDays= [int(day.days) for day in aDays]
    Dftemp=Dftemp.iloc[1:]
    Dftemp['New cases']= aNewCases.div(aDays)
    Dftemp['Incidence']= Dftemp['New cases'].div(Dftemp['Population'])
    lDfIncCHL.append(Dftemp)
    
DfIncCHL=pd.concat(lDfIncCHL)
DfIncCHL.reset_index(inplace=True)

"""
Original dataset: 126819 points, 297 comunas, 427 days
Rediced dataset (minsal rep prod1): 35937 points, 297 comunas, 121 days
dataset from scrapped reports= 110899 datapoints, 297 comunas, 427 days
    
"""

#### Ont

In [ ]:
DfIncOnt= pd.read_csv('../Data/Adherence_Paper/Incidence/Ont/NewCases_Ont.csv', parse_dates=['Date'],
                      index_col=['Date']).stack().reset_index()
DfIncOnt=DfIncOnt[DfIncOnt['level_1']!='_id']
DfIncOnt.columns= ['Date','Region','New cases']
DfIncOnt['Year']=[date.year for date in DfIncOnt['Date']]

DfPopOnt= pd.read_csv('../Data/Adherence_Paper/Incidence/Ont/Pop_Ont.csv', header=9).dropna()
lNamesOntInc=['Algoma_District', 'Brant_County', 'Durham_Region', 'Grey_Bruce', 'Haldimand_Norfolk', 
'Haliburton_Kawartha_Pine_Ridge', 'Halton_Region', 'City_of_Hamilton', 'Hastings_Prince_Edward',
'Huron_Perth', 'Chatham_Kent', 'KFLA', 'Lambton_County', 'Leeds_Grenville_Lanark','Middlesex_London', 
'Niagara_Region', 'North_Bay_Parry_Sound_District', 'Northwestern','City_of_Ottawa', 'Peel_Region','Perth District',
'Peterborough_County_City', 'Porcupine', 'Renfrew_County_and_District', 'Eastern_Ontario', 'Simcoe_Muskoka_District', 
'Sudbury_and_District', 'Thunder_Bay_District', 'Timiskaming', 'Waterloo_Region', 'Wellington_Dufferin_Guelph',
'Windsor_Essex_County', 'York_Region', 'Southwestern','Toronto', ]
 
DfPopOnt['Geography 5 6 7']= lNamesOntInc
DfPopOnt=DfPopOnt.rename(columns={'Geography 5 6 7':'Region'})
DfPopOnt=DfPopOnt.set_index('Region').stack().reset_index().rename(columns={'level_1':'Year',0:'Population'})
DfPopOnt['Year']= DfPopOnt['Year'].astype(int)
DfPopOnt['Population']=[int(sPop.replace(',','')) for sPop in DfPopOnt.Population]

DfIncOnt=DfIncOnt.merge(DfPopOnt, on=['Year','Region'], how='inner').drop(columns='Year')
DfIncOnt['Incidence']= DfIncOnt['New cases'].div(DfIncOnt['Population'].astype(int))

In [ ]:
dicOntPHU={'Algoma_District':['Algoma'] ,'Brant_County':['Brant','Brantford'], 'Chatham_Kent':['Chatham-Kent'],
          'Durham_Region':['Durham'], 'Grey_Bruce':['Grey','Bruce'], 'Porcupine':['Cochrane'], 
          'Wellington_Dufferin_Guelph':['Dufferin','Wellington'], 'Southwestern':['Oxford','Elgin'],
          'Windsor_Essex_County':['Windsor','Essex'],'KFLA':['Frontenac','Lennox and Addington'], 
           'Sudbury_and_District':['Greater Sudbury','Manitoulin','Sudbury'],'Haldimand_Norfolk': ['Haldimand','Norfolk'], 
           'Haliburton_Kawartha_Pine_Ridge':['Haliburton','Kawartha Lakes', 'Northumberland'],
          'Halton_Region':['Halton'], 'City_of_Hamilton':['Hamilton'],
           'Hastings_Prince_Edward':['Hastings', 'Prince Edward'],
          'Huron_Perth':['Huron','Perth'], 'Northwestern':['Kenora','Rainy River'],'Lambton_County':['Lambton'],
          'Leeds_Grenville_Lanark':['Lanark', 'Leeds and Grenville'], 'Middlesex_London':['Middlesex'],
          'Simcoe_Muskoka_District':['Muskoka', 'Simcoe'], 'Niagara_Region':['Niagara'],
          'North_Bay_Parry_Sound_District': ['Nipissing', 'Parry Sound'],'City_of_Ottawa':['Ottawa'],
          'Peel_Region':['Peel'], 'Eastern_Ontario':['Prescott and Russell','Stormont, Dundas and Glengarry'],
           'Renfrew_County_and_District':['Renfrew'], 'Thunder_Bay_District':['Thunder Bay'],
           'Timiskaming':['Timiskaming'],'Toronto': ['Toronto'], 'Waterloo_Region':['Waterloo'], 
           'York_Region':['York'], 'Peterborough_County_City':['Peterborough']}
lDf=[]
for keyreg in dicOntPHU.keys():
    iValReg=len(dicOntPHU[keyreg])
    for valreg in np.arange(iValReg):
        #print (keyreg,dicOntPHU[keyreg][valreg])
        Dftemp=DfIncOnt.set_index('Region').loc[keyreg].reset_index().copy()
        Dftemp['Region']= dicOntPHU[keyreg][valreg]
        lDf.append(Dftemp)
        #print (keyreg)

In [ ]:
DfIncOnt = pd.concat(lDf)

#### ZAF

In [ ]:
DfIncZAF= pd.read_csv('../Data/Adherence_Paper/Incidence/ZAF/covid19za_provincial_cumulative_timeline_confirmed.csv',
                     usecols=[1,2,3,4,5,6,7,8,9,10], parse_dates=['YYYYMMDD'])
DfIncZAF.columns=['Date','Eastern Cape', 'Free State', 'Gauteng', 'KwaZulu-Natal', 'Limpopo',
       'Mpumalanga', 'Northern Cape', 'North West', 'Western Cape']
DfIncZAF=DfIncZAF.set_index('Date')
# need to transform into new daily cases
DfIncZAF=DfIncZAF.loc[idx['2020-03-26':'2022-04-04'],]
DfIncZAF.iloc[1]=(DfIncZAF.iloc[2]-DfIncZAF.iloc[0])/2 +DfIncZAF.iloc[0]
DfIncZAF.loc['2020-04-07']= (DfIncZAF.loc['2020-04-08']-DfIncZAF.loc['2020-04-06'])/2+DfIncZAF.loc['2020-04-06']
DfIncZAF=DfIncZAF.loc[idx['2020-03-27':'2022-04-04'],].sub(DfIncZAF.loc[idx['2020-03-26':'2022-04-03'],].values)
DfIncZAF=DfIncZAF.stack().reset_index().rename(columns={'level_1':'Region',0:'New cases'})
DfIncZAF['Year']=[ date.year for date in DfIncZAF['Date']]

DfPopZAF= pd.read_csv('../Data/Adherence_Paper/Incidence/ZAF/openup-wazi-mid-year-pop-estimates-shaped-for-wazi.csv',
                     index_col=['year'])
DfPopZAF=DfPopZAF.loc[idx[2020:2022],].groupby(['year','geography']).sum(numeric_only=True).reset_index()
DfPopZAF.columns=['Year','Region','Population']

dicTemp= {'EC':'Eastern Cape', 'FS':'Free State', 'GT':'Gauteng','KZN': 'KwaZulu-Natal','LIM': 'Limpopo',
       'MP':'Mpumalanga','NC' :'Northern Cape','NW': 'North West','WC': 'Western Cape'}
DfPopZAF=DfPopZAF[DfPopZAF['Region']!='ZA']
DfPopZAF['Region']= [dicTemp[code] for code in DfPopZAF['Region']]

DfIncZAF=DfIncZAF.merge(DfPopZAF, on=['Year','Region'], how='inner')
DfIncZAF=DfIncZAF.drop(columns=['Year'])
DfIncZAF['Incidence']=DfIncZAF['New cases'].div(DfIncZAF['Population'].values)

## Merging mobility and restrictions

In [ ]:
DfITA=DfResITA.merge(lDfGoogle[0], on=['Region','Date'], how='inner')
DfITA=DfITA.merge(DfIncITA, on=['Region','Date'], how='left')
DfITA=DfITA.set_index(['Region','Date']).sort_index()
DfITA.to_csv('../Data/Adherence_Paper/MobGoogle_Res_ITA.csv')
DfITA

In [ ]:
DfCA=DfResCA.merge(lDfGoogle[2], on=['Region','Date'], how='left')
DfCA= DfCA.merge(DfIncCA, on= ['Date','Region'],how='left')
DfCA=DfCA.set_index(['Region','Date']).sort_index()
DfCA.to_csv('../Data/Adherence_Paper/MobGoogle_Res_CA.csv')
DfCA

In [ ]:
# merge on regions
DfResENG=DfResENG.merge(lDfGoogle[3].drop_duplicates('Region')[['Region']],
               on='Region',how='inner')
# merge on regions and date
DfENG=DfResENG.merge(lDfGoogle[3], left_on=['Region','Date'],right_on=['Region','Date'],how='inner')
DfENG=DfENG.set_index(['Region','Date']).sort_index()
DfENG['Incidence']=0
DfENG.to_csv('../Data/Adherence_Paper/MobGoogle_Res_ENG.csv')
DfENG


In [ ]:
#DfResSCO=DfResSCO.merge(lDfGoogle[4].drop_duplicates('Region')[['Region']],
#               on='Region',how='inner')

DfSCO=DfResSCO.merge(lDfGoogle[4], on=['Region','Date'], how='left')
DfSCO=DfSCO.merge(DfIncSCO, on=['Region','Date'], how='left')
DfSCO=DfSCO.set_index(['Region','Date']).sort_index()
DfSCO.to_csv('../Data/Adherence_Paper/MobGoogle_Res_SCO.csv')
DfSCO

In [ ]:
# google data for supp analysis
#lDfGoogle[1].set_index(['Region','Date']).sort_index().to_csv(('../Data/Adherence_Paper/MobGoogle_Res_CHL.csv'))

# merge with restrictions data
DfCHL=DfMobCHL.merge(DfResCHL, on=['Code','Date'], how='inner')
# merge with incidence data
DfCHL=DfCHL.merge(DfIncCHL, on=['Date', 'Code'], how='left' )

# if using data from product MINSAL
DfCHL=DfCHL.drop(columns=['Region_y']).rename(columns={'Region_x':'Region'})

DfCHL=DfCHL.set_index(['Region','Date']).sort_index()
DfCHL.to_csv(('../../Data/Mobility_Response/Mob_Res_CHL.csv'))


In [ ]:
DfOnt=DfResOnt.merge(lDfGoogle[5], left_on=['Region','Date'],right_on=['Region','Date'], how='left')
DfOnt=DfOnt.reset_index().merge(DfIncOnt, on=['Region','Date'], how='left')
DfOnt=DfOnt.set_index(['Region','Date']).sort_index()
DfOnt.to_csv('../Data/Adherence_Paper/MobGoogle_Res_Ont.csv')
DfOnt

In [ ]:
DfZAF=DfResZAF.drop(columns='Region').merge(lDfGoogle[6], on='Date', how='inner')
DfZAF=DfZAF.merge(DfIncZAF, on=['Date','Region'], how='left')
DfZAF=DfZAF.set_index(['Region','Date']).sort_index()
DfZAF.to_csv('../Data/Adherence_Paper/MobGoogle_Res_ZAF.csv')
DfZAF